# Parte 1 — Exploración de Técnicas de Prompt Engineering

Este notebook explora de forma práctica 8 técnicas de prompt engineering usando Azure AI Foundry.
Cada técnica incluye explicación propia, ejemplo ejecutable, salida del modelo y análisis crítico.

**Técnicas cubiertas:**
1. Zero-Shot Prompting
2. Few-Shot Prompting
3. Chain-of-Thought (CoT)
4. Role Prompting
5. Structured Output Prompting
6. Self-Consistency
7. Least-to-Most Prompting ⭐
8. Generated Knowledge Prompting ⭐

---

## Configuración

Crea un archivo `.env` en el directorio raíz:
```
AZURE_ENDPOINT=https://<tu-proyecto>.openai.azure.com/
AZURE_API_KEY=<tu-api-key>
AZURE_DEPLOYMENT_NAME=<nombre-del-deployment>
AZURE_API_VERSION=2024-02-01
```

In [1]:
!pip install openai python-dotenv --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
import time
import json
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_ENDPOINT"),
    api_key=os.getenv("AZURE_API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION", "2024-02-01")
)

DEPLOYMENT = os.getenv("AZURE_DEPLOYMENT_NAME")

def llamar_modelo(prompt: str, system: str = None, temperature: float = 0.7,
                  max_tokens: int = 600, response_format: dict = None) -> str:
    """Helper reutilizable para todas las llamadas del notebook."""
    mensajes = []
    if system:
        mensajes.append({"role": "system", "content": system})
    mensajes.append({"role": "user", "content": prompt})
    
    kwargs = dict(model=DEPLOYMENT, messages=mensajes,
                  temperature=temperature, max_tokens=max_tokens)
    if response_format:
        kwargs["response_format"] = response_format
    
    r = client.chat.completions.create(**kwargs)
    return r.choices[0].message.content

print(f"Conectado al deployment: {DEPLOYMENT}")

Conectado al deployment: gpt-4o


---
## Técnica 1 — Zero-Shot Prompting

### ¿Qué es?
Es la técnica más básica: le haces una pregunta al modelo directamente, sin darle ningún ejemplo previo. Confías en que el modelo ya sabe hacer la tarea por su entrenamiento. Es rápida y funciona bien para tareas comunes y bien definidas, pero puede fallar en tareas muy específicas o con formatos particulares.

### Cuándo usarla
Tareas genéricas donde el modelo tiene mucho contexto de entrenamiento: traducciones, resúmenes, preguntas de conocimiento general.

In [4]:
prompt_zero_shot = """Clasifica el siguiente texto según su sentimiento: positivo, negativo o neutro.

Texto: 'El despliegue del pipeline tardó 3 horas más de lo esperado y varios jobs fallaron
en producción, pero el equipo reaccionó rápido y al final los datos llegaron correctos.'

Responde solo con: Sentimiento: [positivo/negativo/neutro] y una frase de justificación."""

print("=== ZERO-SHOT: Análisis de sentimiento ===")
print(llamar_modelo(prompt_zero_shot, temperature=0.3))

=== ZERO-SHOT: Análisis de sentimiento ===
Sentimiento: neutro  
Justificación: El texto describe tanto problemas como soluciones, sin inclinarse claramente hacia una valoración positiva o negativa.


### Análisis
El modelo clasifica correctamente sin ejemplos previos. Funciona porque el análisis de sentimiento es una tarea muy común en su entrenamiento. La limitación aparece cuando tienes un vocabulario muy técnico o un dominio muy específico donde el modelo necesita contexto adicional.

---
## Técnica 2 — Few-Shot Prompting

### ¿Qué es?
Le das al modelo 2-5 ejemplos del patrón que quieres que siga antes de hacerle la pregunta real. Los ejemplos actúan como una guía implícita: el modelo aprende el formato, el tono y la lógica que esperas sin que tengas que explicarlo explícitamente. Es especialmente útil cuando el formato de salida es muy concreto o cuando el dominio es específico.

### Cuándo usarla
Cuando necesitas un formato de salida muy preciso, cuando el dominio es especializado, o cuando el zero-shot da resultados inconsistentes.

In [5]:
prompt_few_shot = """Clasifica incidencias de datos según su severidad (P1/P2/P3) y asigna un equipo responsable.

Ejemplos:
Incidencia: 'Pipeline de ventas en tiempo real lleva 2 horas sin datos'
→ Severidad: P1 | Equipo: Data Engineering | Impacto: Crítico - afecta reporting en vivo

Incidencia: 'El dashboard de marketing muestra datos del mes pasado'
→ Severidad: P2 | Equipo: Analytics | Impacto: Medio - datos obsoletos pero no hay pérdida

Incidencia: 'Una columna del catálogo de productos tiene 3 valores nulos'
→ Severidad: P3 | Equipo: Data Quality | Impacto: Bajo - afecta a < 0.01% de registros

Ahora clasifica esta incidencia:
Incidencia: 'El job de ingesta de Azure Data Factory falló a medianoche y no se procesaron
las transacciones de 6 tiendas. Los datos de ayer están incompletos en el DWH.'"""

print("=== FEW-SHOT: Clasificación de incidencias ===")
print(llamar_modelo(prompt_few_shot, temperature=0.3))

=== FEW-SHOT: Clasificación de incidencias ===
**Incidencia:** 'El job de ingesta de Azure Data Factory falló a medianoche y no se procesaron las transacciones de 6 tiendas. Los datos de ayer están incompletos en el DWH.'  
→ **Severidad:** P2  
→ **Equipo:** Data Engineering  
→ **Impacto:** Medio - afecta datos históricos de un subconjunto de tiendas, pero no hay impacto en tiempo real.


### Análisis
Los ejemplos guían al modelo para que use exactamente el formato `Severidad | Equipo | Impacto`. Sin ellos, el modelo respondería en prosa y con un formato diferente cada vez. La clave está en que los ejemplos sean representativos del espectro de casos posibles (P1, P2, P3). Si todos los ejemplos son P1, el modelo tenderá a sobreclasificar.

---
## Técnica 3 — Chain-of-Thought (CoT)

### ¿Qué es?
Le pides explícitamente al modelo que piense paso a paso antes de dar la respuesta final. En vez de saltar directamente a la conclusión, el modelo desglosa el problema en pasos intermedios. Esto reduce los errores en razonamiento lógico, matemático o técnico porque cada paso actúa como un checkpoint. Es como pedirle a alguien que muestre su trabajo en lugar de darte solo el resultado.

### Cuándo usarla
Problemas que requieren múltiples pasos lógicos, estimaciones, comparaciones técnicas o decisiones complejas.

In [6]:
prompt_cot = """Tengo que decidir entre tres opciones de almacenamiento para un data lake en Azure.
Razona paso a paso antes de dar tu recomendación.

Contexto:
- Volumen: 50 TB actuales, crecimiento estimado 20% anual
- Acceso: 80% de los datos se consultan raramente (archivado), 20% con acceso frecuente
- Equipo: 2 data engineers, no hay administradores de infraestructura dedicados
- Budget: no definido, pero hay presión de costes

Opciones:
A) Azure Data Lake Storage Gen2 con tier caliente para todo
B) ADLS Gen2 con lifecycle management (hot/cool/archive automático)
C) Azure Blob Storage con tier frío para todo

Piensa paso a paso: analiza cada opción, considera los trade-offs y luego da tu recomendación final."""

print("=== CHAIN-OF-THOUGHT: Decisión de arquitectura ===")
print(llamar_modelo(prompt_cot, temperature=0.4, max_tokens=800))

=== CHAIN-OF-THOUGHT: Decisión de arquitectura ===
Vamos a analizar cada opción paso a paso considerando el contexto proporcionado:

---

### **Opción A: Azure Data Lake Storage Gen2 con tier caliente para todo**

**Ventajas:**
1. **Simplicidad de gestión:** No requiere configuraciones adicionales para mover datos entre tiers de almacenamiento.
2. **Rendimiento:** El tier caliente ofrece el mejor rendimiento en términos de latencia y velocidad de acceso, lo cual puede ser beneficioso para los datos que se consultan frecuentemente (20% del total).

**Desventajas:**
1. **Costo elevado:** El tier caliente es el más caro de las opciones de almacenamiento en Azure. Dado que el 80% de los datos se consultan raramente, mantenerlos en este tier sería un uso ineficiente del presupuesto.
2. **Escalabilidad:** A medida que el volumen de datos crezca (20% anual), los costos asociados al tier caliente se incrementarán considerablemente, ejerciendo más presión sobre el presupuesto.

**Conclusión par

### Análisis
El CoT es especialmente valioso aquí porque la decisión depende de múltiples variables que interactúan entre sí. Sin el razonamiento paso a paso, el modelo podría dar una respuesta genérica. Con CoT, el proceso de eliminación es visible y auditable. La desventaja es que consume más tokens y es más lento. Para decisiones simples, es overkill.

---
## Técnica 4 — Role Prompting

### ¿Qué es?
Le asignas al modelo un rol específico con experiencia, perspectiva o estilo concreto. Cuando el modelo asume un rol, ajusta el vocabulario, el nivel técnico, el tono y el enfoque de la respuesta a ese perfil. No es solo una cuestión de estilo: el rol activa patrones de conocimiento específicos del entrenamiento.

### Cuándo usarla
Cuando necesitas una perspectiva experta específica, cuando el público objetivo es muy concreto (un CEO vs un data engineer), o cuando quieres simular una revisión técnica o una entrevista.

In [7]:
descripcion_proyecto = """
Proyecto: Pipeline de ingesta en tiempo real de transacciones de e-commerce.
Stack: Azure Event Hubs → Azure Stream Analytics → Azure SQL Database.
Latencia actual: 800ms p95. Objetivo: < 200ms p95.
Coste mensual actual: 3,200€.
"""

roles = [
    ("Arquitecto de soluciones Azure senior con 10 años de experiencia en sistemas de alta disponibilidad",
     "Revisa el diseño técnico y señala los riesgos de arquitectura."),
    ("CFO de una empresa de tecnología preocupado por los costes de cloud",
     "Evalúa si el gasto es justificado y qué optimizaciones de coste propones.")
]

print("=== ROLE PROMPTING: Dos perspectivas sobre el mismo proyecto ===")
for rol, pregunta in roles:
    print(f"\n--- ROL: {rol[:60]}... ---")
    respuesta = llamar_modelo(
        f"{pregunta}\n\nProyecto:\n{descripcion_proyecto}",
        system=f"Eres un {rol}. Responde desde esa perspectiva específica.",
        temperature=0.5, max_tokens=400
    )
    print(respuesta)
    time.sleep(1)

=== ROLE PROMPTING: Dos perspectivas sobre el mismo proyecto ===

--- ROL: Arquitecto de soluciones Azure senior con 10 años de experie... ---
Con base en el diseño técnico presentado, aquí está mi análisis de los riesgos de arquitectura y posibles áreas de mejora:

---

### **Riesgos Identificados**

#### **1. Latencia insuficiente para alcanzar el objetivo (< 200ms p95)**
   - **Problema**: La latencia actual (800ms p95) es significativamente superior al objetivo (< 200ms p95). Esto sugiere que hay cuellos de botella en el pipeline.
   - **Posibles causas**:
     - **Azure Stream Analytics (ASA)**: ASA es un servicio poderoso, pero su procesamiento puede tener latencias superiores a los 200ms dependiendo de la complejidad de las consultas, el tamaño de los eventos y la configuración del job.
     - **Azure SQL Database**: Las operaciones de escritura en bases de datos relacionales como Azure SQL Database pueden introducir latencias adicionales, especialmente si hay índices complejos,

### Análisis
El mismo proyecto recibe respuestas completamente diferentes según el rol. El arquitecto se centra en latencia y resiliencia; el CFO en coste-beneficio. Esto demuestra que el role prompting no es cosmético: activa diferentes patrones de razonamiento. Es una técnica muy útil para generar documentación que necesita ser leída por diferentes stakeholders.

---
## Técnica 5 — Structured Output Prompting

### ¿Qué es?
Le indicas explícitamente al modelo el formato exacto de la respuesta: JSON, tabla Markdown, XML, lista numerada con campos específicos, etc. Es diferente al few-shot en que no das ejemplos de contenido, sino solo la estructura vacía o el schema que debe rellenar. Es fundamental para integrar las respuestas del modelo en sistemas de software.

### Cuándo usarla
Siempre que la salida del modelo vaya a ser procesada por código downstream: APIs, bases de datos, dashboards.

In [8]:
prompt_structured = """Analiza el siguiente mensaje de error de un pipeline de datos y devuelve
el análisis ÚNICAMENTE en el siguiente formato JSON, sin texto adicional:

{
  "error_tipo": "string",
  "causa_raiz": "string",
  "severidad": "critica|alta|media|baja",
  "componente_afectado": "string",
  "pasos_resolucion": ["string"],
  "tiempo_resolucion_estimado": "string",
  "requiere_escalado": true
}

Error:
"SparkException: Job aborted due to stage failure: Task 45 in stage 12.0 failed 4 times,
most recent failure: Lost task 45.3 in stage 12.0 (TID 1823): java.lang.OutOfMemoryError:
GC overhead limit exceeded at org.apache.spark.sql.execution.joins.HashedRelation"""

print("=== STRUCTURED OUTPUT: Diagnóstico de error Spark ===")
raw = llamar_modelo(prompt_structured, temperature=0.2,
                    response_format={"type": "json_object"})
print(raw)

# Validar que es JSON válido y acceder a los campos
try:
    parsed = json.loads(raw)
    print(f"\n✅ JSON válido. Severidad detectada: {parsed.get('severidad')}")
    print(f"Requiere escalado: {parsed.get('requiere_escalado')}")
    print(f"Pasos de resolución: {len(parsed.get('pasos_resolucion', []))} pasos")
except json.JSONDecodeError as e:
    print(f"❌ JSON inválido: {e}")

=== STRUCTURED OUTPUT: Diagnóstico de error Spark ===
{
  "error_tipo": "SparkException",
  "causa_raiz": "GC overhead limit exceeded debido a un OutOfMemoryError en una operación de join en Spark.",
  "severidad": "alta",
  "componente_afectado": "Spark SQL - Join Operation",
  "pasos_resolucion": [
    "Incrementar la memoria asignada al ejecutor en la configuración de Spark.",
    "Optimizar la consulta para reducir el tamaño de los datos procesados en el join.",
    "Habilitar broadcast join si el tamaño de una de las tablas es pequeño.",
    "Revisar y ajustar los parámetros de configuración de GC (Garbage Collection) en Spark."
  ],
  "tiempo_resolucion_estimado": "2-4 horas",
  "requiere_escalado": true
}

✅ JSON válido. Severidad detectada: alta
Requiere escalado: True
Pasos de resolución: 4 pasos


### Análisis
Esta técnica es imprescindible en producción. La combinación de instrucciones de schema en el prompt + `response_format: json_object` es más robusta que confiar solo en el prompt. El truco está en que el schema sea específico pero no demasiado anidado (estructuras muy profundas aumentan los errores). Para casos críticos, añadir validación con Pydantic como capa adicional.

---
## Técnica 6 — Self-Consistency

### ¿Qué es?
En vez de confiar en una sola respuesta del modelo, generas varias respuestas independientes para el mismo problema (normalmente con temperatura alta para fomentar variabilidad) y luego seleccionas la respuesta más frecuente o la haces votar. Es una forma de reducir la varianza del modelo usando "sabiduría de la multitud" dentro del propio modelo. Más costoso pero más fiable para problemas donde el error tiene consecuencias altas.

### Cuándo usarla
Decisiones críticas donde un error del modelo tiene consecuencias reales, preguntas técnicas con respuesta objetiva, clasificaciones donde la consistencia importa más que la velocidad.

In [9]:
from collections import Counter

prompt_consistencia = """Analiza este query SQL y determina si tiene un problema de rendimiento.
Responde SOLO con: SÍ o NO, seguido de una frase de justificación.

Query:
SELECT c.nombre, COUNT(p.id) as total_pedidos, SUM(p.importe) as total_importe
FROM clientes c
LEFT JOIN pedidos p ON c.id = p.cliente_id
WHERE YEAR(p.fecha_pedido) = 2024
AND p.estado NOT IN (SELECT estado FROM estados_cancelados)
GROUP BY c.nombre
ORDER BY total_importe DESC"""

print("=== SELF-CONSISTENCY: Análisis de query SQL (5 ejecuciones) ===")

respuestas = []
for i in range(5):
    r = llamar_modelo(prompt_consistencia, temperature=0.8, max_tokens=150)
    respuestas.append(r)
    veredicto = "SÍ" if r.strip().upper().startswith("SÍ") or r.strip().upper().startswith("SI") else "NO"
    print(f"Ejecución {i+1}: {veredicto} — {r[:80]}..." if len(r) > 80 else f"Ejecución {i+1}: {r}")
    time.sleep(0.5)

# Determinar respuesta mayoritaria
veredictos = ["SÍ" if r.strip().upper().startswith("SÍ") or r.strip().upper().startswith("SI") else "NO"
              for r in respuestas]
conteo = Counter(veredictos)
ganador = conteo.most_common(1)[0]
print(f"\n📊 Resultado por votación: {ganador[0]} ({ganador[1]}/5 votos)")
print(f"Confianza: {ganador[1]/5*100:.0f}%")

=== SELF-CONSISTENCY: Análisis de query SQL (5 ejecuciones) ===
Ejecución 1: SÍ — SÍ, el uso de una subconsulta correlacionada en la cláusula `NOT IN` puede gener...
Ejecución 2: SÍ — SÍ, el problema radica en que la cláusula `WHERE YEAR(p.fecha_pedido) = 2024` se...
Ejecución 3: SÍ — SÍ, el uso de la función `YEAR()` en la cláusula `WHERE` sobre la columna `p.fec...
Ejecución 4: SÍ — SÍ. La subconsulta en la cláusula `NOT IN` puede causar problemas de rendimiento...
Ejecución 5: SÍ — SÍ, el uso de una subconsulta en la cláusula `NOT IN` y la función `YEAR` en la ...

📊 Resultado por votación: SÍ (5/5 votos)
Confianza: 100%


### Análisis
La self-consistency aumenta la fiabilidad a costa de múltiples llamadas (5x coste en tokens). En este caso el query tiene problemas obvios (función en WHERE que impide uso de índice, subquery correlacionada) por lo que el modelo debería ser consistente. Donde más valor aporta es en problemas donde la respuesta no es obvia y el modelo tiene incertidumbre interna.

---
## Técnica 7 — Least-to-Most Prompting ⭐

### ¿Qué es?
Es una extensión del CoT donde primero le pides al modelo que descomponga el problema en subproblemas ordenados de menor a mayor complejidad, y luego los resuelve secuencialmente usando las soluciones anteriores como contexto. Es especialmente poderoso para problemas que tienen dependencias entre pasos: no puedes resolver el paso 3 sin haber resuelto el 2.

> Esta técnica no aparece documentada en TEORIA.md. Fuente: Wei et al. (2022), "Least-to-Most Prompting Enables Complex Reasoning in Large Language Models" — https://arxiv.org/abs/2205.10625

### Cuándo usarla
Diseño de arquitecturas complejas, debugging de sistemas con múltiples capas, planificación de proyectos técnicos donde cada fase depende de la anterior.

In [10]:
# Paso 1: Descomposición del problema
problema = """Necesito migrar una base de datos SQL Server on-premise de 2 TB con 500 usuarios
activos a Azure SQL Database, con tiempo de inactividad máximo de 2 horas y sin pérdida de datos."""

print("=== LEAST-TO-MOST: Migración de base de datos ===")
print("--- PASO 1: Descomposición del problema ---")

prompt_descomposicion = f"""Descompón este problema en subproblemas ordenados de menor a mayor complejidad.
Lista SOLO los subproblemas numerados, sin resolverlos todavía.

Problema: {problema}"""

subproblemas = llamar_modelo(prompt_descomposicion, temperature=0.3, max_tokens=400)
print(subproblemas)

time.sleep(1)

# Paso 2: Resolución secuencial con contexto acumulado
print("\n--- PASO 2: Resolución secuencial ---")

prompt_resolucion = f"""Problema original: {problema}

Subproblemas identificados:
{subproblemas}

Ahora resuelve cada subproblema en orden, usando las soluciones anteriores como base para las siguientes.
Para cada subproblema, indica cómo su solución afecta o facilita los siguientes pasos."""

solucion = llamar_modelo(prompt_resolucion, temperature=0.4, max_tokens=900)
print(solucion)

=== LEAST-TO-MOST: Migración de base de datos ===
--- PASO 1: Descomposición del problema ---
1. Evaluar la compatibilidad de la base de datos on-premise con Azure SQL Database.  
2. Analizar el uso y las dependencias actuales de la base de datos (triggers, stored procedures, índices, etc.).  
3. Estimar el tiempo necesario para la migración de datos y validar si cumple con el límite de 2 horas.  
4. Seleccionar la herramienta o método de migración más adecuado para el escenario.  
5. Diseñar un plan de migración detallado, incluyendo pasos para minimizar el tiempo de inactividad.  
6. Configurar un entorno de prueba en Azure SQL Database para realizar simulaciones.  
7. Realizar una migración de prueba para identificar posibles problemas.  
8. Optimizar la base de datos en Azure para el rendimiento esperado.  
9. Planificar y comunicar el cronograma de migración a los usuarios afectados.  
10. Configurar mecanismos de respaldo y recuperación para garantizar la integridad de los datos.

### Análisis
El enfoque de dos fases (descomponer → resolver) produce planes más coherentes que pedir la solución directamente. Al resolver en orden, el modelo no pierde de vista las dependencias entre pasos. La diferencia con el CoT estándar es que aquí el modelo primero mapea el territorio antes de recorrerlo. Ideal para planificaciones técnicas que luego se van a ejecutar realmente.

---
## Técnica 8 — Generated Knowledge Prompting ⭐

### ¿Qué es?
Antes de pedir la respuesta final, le pides al modelo que genere conocimiento relevante sobre el tema. Luego usas ese conocimiento generado como contexto adicional para la pregunta real. Es como forzar al modelo a activar y articular su memoria de trabajo antes de razonar. El conocimiento generado actúa como un andamio que mejora la calidad del razonamiento posterior.

> Esta técnica no aparece en TEORIA.md. Fuente: Liu et al. (2022), "Generated Knowledge Prompting for Commonsense Reasoning" — https://arxiv.org/abs/2110.08387

### Cuándo usarla
Cuando el modelo necesita conocimiento muy específico para responder bien, cuando la respuesta directa tiende a ser superficial, o cuando quieres asegurarte de que el modelo considera todos los factores relevantes.

In [11]:
pregunta_real = """¿Debería usar Microsoft Fabric o Azure Databricks para una empresa de
retail que ya tiene licencias de Microsoft 365 y Power BI Premium, con un equipo de datos
de 5 personas sin experiencia previa en Spark?"""

print("=== GENERATED KNOWLEDGE: Decisión Fabric vs Databricks ===")

# Fase 1: Generar conocimiento relevante
print("--- FASE 1: Generación de conocimiento relevante ---")
prompt_conocimiento = """Genera un resumen técnico objetivo sobre estos 5 aspectos clave
para comparar Microsoft Fabric y Azure Databricks:
1. Curva de aprendizaje y prerequisitos técnicos
2. Integración con el ecosistema Microsoft 365 y Power BI
3. Coste para equipos pequeños (< 10 personas)
4. Capacidades de procesamiento Spark y Big Data
5. Madurez y soporte del producto en 2024-2025

Sé objetivo y equilibrado, sin favorecer ninguna opción."""

conocimiento = llamar_modelo(prompt_conocimiento, temperature=0.3, max_tokens=600)
print(conocimiento)

time.sleep(1)

# Fase 2: Usar el conocimiento generado para responder
print("\n--- FASE 2: Respuesta informada usando el conocimiento generado ---")
prompt_final = f"""Usando el siguiente análisis técnico como contexto:

{conocimiento}

Responde esta pregunta con una recomendación concreta y justificada:
{pregunta_real}"""

recomendacion = llamar_modelo(prompt_final, temperature=0.4, max_tokens=400)
print(recomendacion)

=== GENERATED KNOWLEDGE: Decisión Fabric vs Databricks ===
--- FASE 1: Generación de conocimiento relevante ---
### Resumen técnico objetivo: Comparativa entre Microsoft Fabric y Azure Databricks

#### 1. **Curva de aprendizaje y prerequisitos técnicos**
   - **Microsoft Fabric**: Diseñado para usuarios con experiencia básica en herramientas de Microsoft 365 y Power BI, ofrece una experiencia más accesible para analistas de datos y usuarios de negocio. Su interfaz unificada y enfoque "low-code/no-code" reduce la curva de aprendizaje, aunque puede ser limitada para usuarios avanzados que necesiten personalización profunda o programación compleja.
   - **Azure Databricks**: Tiene una curva de aprendizaje más pronunciada, ya que requiere conocimientos en lenguajes como Python, Scala o SQL, así como experiencia con Apache Spark y conceptos de Big Data. Es más adecuado para equipos técnicos con experiencia en ingeniería de datos y ciencia de datos.

#### 2. **Integración con el ecosistema M

### Análisis
La diferencia entre la respuesta directa y la respuesta con conocimiento previo generado es notable: la segunda es más matizada y considera aspectos que de otro modo el modelo podría omitir (como la integración con licencias existentes o la curva de aprendizaje). El coste es el doble de llamadas, pero la calidad justifica el gasto en decisiones importantes.

---
## Sección de Casos de Uso Reales

### Caso 1 — Generación automática de documentación técnica de pipelines

**Técnicas aplicadas:** Role Prompting + Structured Output + Few-Shot  
**Justificación:** El role prompting asegura el tono técnico correcto; el structured output garantiza un formato estándar que pueda integrarse en un wiki o Confluence; el few-shot establece el nivel de detalle esperado.

In [12]:
pipeline_info = """
Nombre: ingesta_ventas_daily
Tecnología: Azure Data Factory + Databricks
Frecuencia: Diaria a las 02:00 UTC
Origen: Azure SQL Database (tabla: transactions, ~2M filas/día)
Destino: ADLS Gen2 (formato Delta Lake, particionado por fecha)
Transformaciones: deduplicación, normalización de monedas, cálculo de métricas diarias
SLA: datos disponibles antes de las 06:00 UTC
Alertas: email a data-team@empresa.com si falla
"""

ejemplo_doc = """
## Pipeline: clientes_sync_hourly
**Propósito:** Sincronización horaria del catálogo de clientes desde CRM.
**SLA:** Datos disponibles en < 15 min desde la ejecución.
**Dependencias:** Requiere que el job `crm_export` haya completado.
**Monitorización:** Alertas vía PagerDuty si latencia > 20 min.
"""

prompt_doc = f"""Ejemplo de documentación de pipeline:
{ejemplo_doc}

Genera documentación técnica en el mismo formato para este pipeline:
{pipeline_info}

Incluye: Propósito, Dependencias, SLA, Monitorización, Procedimiento de recuperación ante fallos."""

print("=== CASO 1: Documentación de pipeline ===")
print(llamar_modelo(
    prompt_doc,
    system="Eres un data engineer senior especializado en documentación técnica. Escribes documentación clara, concisa y orientada a operaciones.",
    temperature=0.4, max_tokens=600
))

=== CASO 1: Documentación de pipeline ===
## Pipeline: ingesta_ventas_daily

**Propósito:**  
Realizar la ingesta diaria de datos de transacciones desde Azure SQL Database, aplicando transformaciones clave (deduplicación, normalización de monedas, cálculo de métricas diarias) y almacenando los resultados en ADLS Gen2 en formato Delta Lake. Este pipeline alimenta los reportes financieros diarios y dashboards de ventas.

---

**Dependencias:**  
- El origen de datos (Azure SQL Database, tabla `transactions`) debe estar actualizado y accesible.  
- Requiere que el job de preprocesamiento en Azure SQL Database (si aplica) haya finalizado antes de las 02:00 UTC.  
- Conexión estable entre Azure Data Factory y Databricks.  

---

**SLA:**  
Los datos procesados deben estar disponibles en el destino (ADLS Gen2) antes de las 06:00 UTC diariamente.  

---

**Monitorización:**  
- Alertas configuradas para notificar fallos en la ejecución del pipeline.  
- En caso de error, se envía un correo au

### Caso 2 — Revisión automática de código SQL con análisis de rendimiento

**Técnicas aplicadas:** Chain-of-Thought + Structured Output + Role Prompting  
**Justificación:** El CoT fuerza al modelo a analizar el query sistemáticamente antes de dar recomendaciones; el structured output facilita integrarlo en un pipeline de CI/CD de SQL; el rol de DBA senior activa conocimiento específico de optimización.

In [13]:
query_revision = """
SELECT 
    u.email,
    u.nombre,
    (SELECT COUNT(*) FROM pedidos p WHERE p.usuario_id = u.id) as total_pedidos,
    (SELECT SUM(importe) FROM pedidos p WHERE p.usuario_id = u.id 
     AND YEAR(p.fecha) = YEAR(GETDATE())) as importe_anual,
    (SELECT TOP 1 p.fecha FROM pedidos p WHERE p.usuario_id = u.id 
     ORDER BY p.fecha DESC) as ultimo_pedido
FROM usuarios u
WHERE u.activo = 1
ORDER BY importe_anual DESC
"""

prompt_revision = f"""Analiza este query SQL paso a paso:
1. Identifica cada problema de rendimiento
2. Explica por qué cada problema impacta el rendimiento
3. Propón la versión optimizada
4. Estima el impacto de mejora (bajo/medio/alto)

Responde en JSON con el formato:
{{
  "problemas": [{{"tipo": string, "descripcion": string, "linea": string}}],
  "query_optimizado": string,
  "indices_recomendados": [string],
  "impacto_mejora": "bajo|medio|alto",
  "justificacion": string
}}

Query a analizar:
{query_revision}"""

print("=== CASO 2: Revisión de query SQL ===")
resultado = llamar_modelo(
    prompt_revision,
    system="Eres un DBA senior especializado en optimización de queries SQL Server y Azure SQL. Tienes criterio práctico y no teórico.",
    temperature=0.2,
    response_format={"type": "json_object"},
    max_tokens=800
)
print(resultado)
parsed = json.loads(resultado)
print(f"\n📊 Problemas encontrados: {len(parsed.get('problemas', []))}")
print(f"Impacto de mejora: {parsed.get('impacto_mejora')}")

=== CASO 2: Revisión de query SQL ===
{
  "problemas": [
    {
      "tipo": "Subquery repetitiva",
      "descripcion": "El query utiliza tres subqueries independientes sobre la tabla 'pedidos', lo que genera múltiples accesos a la misma tabla para cada fila de 'usuarios'. Esto impacta negativamente el rendimiento debido a la redundancia en el cálculo.",
      "linea": "3-7"
    },
    {
      "tipo": "Función en la cláusula WHERE",
      "descripcion": "La condición 'YEAR(p.fecha) = YEAR(GETDATE())' en la subquery de 'importe_anual' obliga a SQL Server a aplicar la función YEAR a cada fila de la tabla 'pedidos', lo que impide el uso eficiente de índices sobre 'fecha'.",
      "linea": "5"
    },
    {
      "tipo": "Ordenamiento en subquery",
      "descripcion": "La subquery para 'ultimo_pedido' utiliza 'ORDER BY p.fecha DESC' junto con 'TOP 1', lo que puede ser costoso si no hay un índice adecuado sobre 'fecha'.",
      "linea": "7"
    },
    {
      "tipo": "Ordenamiento principa

---
## Conclusiones Personales

### ¿Qué técnicas usaría en proyectos reales?

**Las que usaría siempre:**
- **Structured Output**: imprescindible para cualquier integración de LLM en sistemas de software. Sin formato controlado, las salidas son impredecibles.
- **Role Prompting**: el cambio en calidad de respuesta es notable y prácticamente no tiene coste adicional.
- **Few-Shot**: para cualquier tarea que necesite un formato de salida consistente en múltiples ejecuciones.

**Las que usaría en casos específicos:**
- **Chain-of-Thought**: solo para problemas que realmente lo requieren. Para preguntas simples añade ruido.
- **Least-to-Most**: planificaciones técnicas complejas o diseños de arquitectura.
- **Self-Consistency**: solo cuando el coste de un error del modelo es alto y el coste de tokens es tolerable.
- **Generated Knowledge**: cuando la respuesta directa tiende a ser demasiado genérica.

**Aprendizaje principal:** La combinación de técnicas es más poderosa que cualquier técnica individual. Role + Structured + CoT juntos producen resultados notablemente superiores a cada uno por separado.